## Sentiment Analysis

In [7]:
%pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [9]:
import soundfile as sf

In [10]:
import torch
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
from IPython.display import Audio


In [11]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.


### How it works:

STEP 1: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


STEP 2: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [12]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device = "mps")

result = my_simple_sentiment_analyzer("I am super excited to be on the may to master LLM!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps


[{'label': 'POSITIVE', 'score': 0.9997201561927795}]


In [13]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

[{'label': 'POSITIVE', 'score': 0.9008290767669678}]


In [14]:

better_sentiment = pipeline("sentiment-analysis", model = "nlptown/bert-base-multilingual-uncased-sentiment", device = "mps")

result = better_sentiment("I should be more excited to be on the way to LLM mastery!")
print(result)

Device set to use mps


[{'label': '3 stars', 'score': 0.3901219367980957}]


In [15]:
# ner = Named Entity Recognition

ner = pipeline("ner", device = "mps")

result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Pradeep Kumar")

for entity in result:
    print(entity)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps


{'entity': 'I-ORG', 'score': np.float32(0.9995179), 'index': 1, 'word': 'AI', 'start': 0, 'end': 2}
{'entity': 'I-ORG', 'score': np.float32(0.9973049), 'index': 2, 'word': 'Engineers', 'start': 3, 'end': 12}
{'entity': 'I-ORG', 'score': np.float32(0.9006538), 'index': 11, 'word': 'Hu', 'start': 59, 'end': 61}
{'entity': 'I-ORG', 'score': np.float32(0.8126666), 'index': 12, 'word': '##gging', 'start': 61, 'end': 66}
{'entity': 'I-ORG', 'score': np.float32(0.975134), 'index': 13, 'word': '##F', 'start': 66, 'end': 67}
{'entity': 'I-ORG', 'score': np.float32(0.9483948), 'index': 14, 'word': '##ace', 'start': 67, 'end': 70}
{'entity': 'I-MISC', 'score': np.float32(0.8253285), 'index': 16, 'word': 'Google', 'start': 74, 'end': 80}
{'entity': 'I-MISC', 'score': np.float32(0.7077021), 'index': 17, 'word': 'Cola', 'start': 81, 'end': 85}
{'entity': 'I-MISC', 'score': np.float32(0.6795153), 'index': 18, 'word': '##b', 'start': 85, 'end': 86}
{'entity': 'I-PER', 'score': np.float32(0.9996828), '

In [16]:
# Q&A with context

question = "What are the huggingface pipelines?"
context = "Pipelines are high level API for inference of LLMs with common tasks"

my_pipeline = pipeline("question-answering", device = "mps")
result = my_pipeline(question = question, context = context)

print(result)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps


{'score': 0.40045779943466187, 'start': 14, 'end': 68, 'answer': 'high level API for inference of LLMs with common tasks'}


In [17]:
# Text Summarization

my_pipeline = pipeline("summarization", device = "mps")

text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

result = my_pipeline(text)

print(result)
print("-----------------")

result = my_pipeline(text, max_length=50, min_length=25, do_sample=False)

print(result)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps
Your max_length is set to 142, but your input_length is only 100. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)


[{'summary_text': " The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing . It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering . It's an extremely popular library that's widely used by the open-source data science community ."}]
-----------------
[{'summary_text': ' The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing . It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering .'}]


In [18]:
# Translation (English to French)

translator = pipeline("translation_en_to_fr", device = "mps")

text = "My name is Pradeep Kumar"

result = translator(text)

print(result)
print(result[0]['translation_text'])

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps


[{'translation_text': 'Mon nom est Pradeep Kumar'}]
Mon nom est Pradeep Kumar


In [19]:
# Translation (English to Spanish)
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending

%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [20]:
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device = "mps")

text = "My name is Pradeep Kumar"

result = translator(text)

print(result)

/Users/pradeepkumar/miniconda3/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps


[{'translation_text': 'Mi nombre es Pradeep Kumar.'}]


In [22]:
# Classification

classifier = pipeline("zero-shot-classification", device="mps")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps


{'sequence': "Hugging Face's Transformers library is amazing!", 'labels': ['technology', 'sports', 'politics'], 'scores': [0.949384331703186, 0.03224988281726837, 0.018365776166319847]}


In [23]:
# Text Generation

generator = pipeline("text-generation", device="mps")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


If there's one thing I want you to remember about using HuggingFace pipelines, it's that the pipeline you use is also the only one that actually works with the code. So if you're going to use HuggingFace, you should use the standard-compliant HuggingFace API.

The most important thing is that you actually use the HuggingFace Pipeline as opposed to the standard-compliant HuggingFace API. You should work with the standard-compliant HuggingFace API in the same way as you would with the regular HuggingFace API.

Here's a simple example of how you can work with the standard-compliant HuggingFace API:

import { _. pipeline } from'react'; import { _. push } from'react-native'; import { _. push } from'react-router'; import { _. push } from'react-dom'; import { _. push } from'react-dom-model'; import { _. push } from'react-dom-model-css'; import { _. push } from'react-dom-css'; import { _. push } from'react-dom-css-repeat'; import { _. push } from'react-dom-css-repeat'; import { _. push } from'